# CBIS-DDSM preprocessing review

Visual and numeric check of the preprocessing of CBIS-DDSM: the breast crop box on
the original images, the preprocessed images (crop, rescale, flip) with their
masks, and the mapping of a mask back onto the original image. Run the steps
first:

```bash
uv run python -m mammo_seg.data.manifest --dataset cbis-ddsm
uv run python -m mammo_seg.data.masks --dataset cbis-ddsm
uv run python -m mammo_seg.data.preprocess_dataset --dataset cbis-ddsm
```

Outputs are stripped before commit (`nbstripout`).

The questions, as for INbreast (`03_preprocessing_review.ipynb`): does every box
contain the whole breast and little else, despite the bright film edges and
labels of the scanned films (the film edges are cleared, then everything outside
the breast is set to 0); is every chest wall on the left after preprocessing,
although the orientation of the originals does not follow the laterality; and
does each mask still sit on its mass?


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from scipy import ndimage

from mammo_seg.data.manifest import read_manifest
from mammo_seg.data.masks import mask_path, read_mask
from mammo_seg.data.paths import CBIS_DDSM_PROCESSED_DIR, CBIS_DDSM_RAW_DIR
from mammo_seg.data.preprocess_dataset import (
    image_path,
    processed_mask_path,
    read_image,
    read_record,
)
from mammo_seg.data.preprocessing import restore_mask


def find_repo_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / "pyproject.toml").exists():
            return parent
    raise FileNotFoundError("pyproject.toml not found above " + str(start))


ROOT = find_repo_root(Path.cwd().resolve())
RAW = ROOT / CBIS_DDSM_RAW_DIR
MASKS = ROOT / CBIS_DDSM_PROCESSED_DIR / "masks"
OUT = ROOT / CBIS_DDSM_PROCESSED_DIR / "preprocessed"

manifest = read_manifest(ROOT / CBIS_DDSM_PROCESSED_DIR / "manifest.json")
record = read_record(OUT / "preprocessing.json")
settings = record.settings
print(settings)
print(len(manifest.images), "images,", len(record.images), "preprocessed")

## Crop boxes on the original images

The green rectangle is the crop box (with its margin) on the original JPEG,
downsampled for display; the title says whether the image was flipped (its chest
wall was found on the right). `PAGE` selects 30 images at a time.


In [ ]:
PER_PAGE = 30
STEP = 16  # display downscale of the originals


def small_original(image) -> np.ndarray:
    with Image.open(RAW / image.image_path) as original:
        original.draft("L", (original.width // STEP, original.height // STEP))
        return np.asarray(original.resize((image.cols // STEP, image.rows // STEP)))


def show_boxes(images: list) -> None:
    columns = 6
    rows = -(-len(images) // columns)
    fig, axes = plt.subplots(
        rows, columns, figsize=(2.6 * columns, 3.6 * rows), squeeze=False
    )
    for ax in axes.ravel():
        ax.axis("off")
    for ax, image in zip(axes.ravel(), images, strict=False):
        ax.imshow(small_original(image), cmap="gray", vmin=0, vmax=255)
        params = record.images[image.file_id]
        box = params.box
        top, bottom, left, right = (
            v / STEP for v in (box.top, box.bottom, box.left, box.right)
        )
        ax.plot(
            [left, right, right, left, left],
            [top, top, bottom, bottom, top],
            color="lime",
            linewidth=1,
        )
        flipped = "flipped" if params.flipped else ""
        ax.set_title(f"{image.laterality}{image.view} {flipped}", fontsize=7)
    plt.tight_layout()
    plt.show()


PAGE = 0
pages = -(-len(manifest.images) // PER_PAGE)
print(f"page {PAGE + 1} of {pages}")
show_boxes(list(manifest.images[PAGE * PER_PAGE : (PAGE + 1) * PER_PAGE]))

## Preprocessed images with their masks

The stored images (8-bit values in 16-bit files), as the model will see them
before normalization, with the mask outline in red. The chest wall should be on
the left in every image.


In [ ]:
def show_processed(images: list) -> None:
    columns = 6
    rows = -(-len(images) // columns)
    fig, axes = plt.subplots(
        rows, columns, figsize=(2.6 * columns, 4.2 * rows), squeeze=False
    )
    for ax in axes.ravel():
        ax.axis("off")
    for ax, image in zip(axes.ravel(), images, strict=False):
        pixels = read_image(image_path(OUT, image.file_id))
        mask = read_mask(processed_mask_path(OUT, image.file_id))
        ax.imshow(pixels, cmap="gray", vmin=0, vmax=settings.max_intensity)
        ax.contour(mask, levels=[0.5], colors="red", linewidths=0.8)
        ax.set_title(f"{image.laterality}{image.view}", fontsize=7)
    plt.tight_layout()
    plt.show()


print(f"page {PAGE + 1} of {pages}")
show_processed(list(manifest.images[PAGE * PER_PAGE : (PAGE + 1) * PER_PAGE]))

## Numeric checks on all images

- **Orientation:** after preprocessing, the left third of the image should hold
  more tissue than the right third (chest wall on the left).
- **Box:** the boxes that cover almost the whole image, the share of the
  cropped image that is tissue, and the size of the results.
- **Masks:** every mask is brighter than a ring of tissue around it, and a mask
  restored onto the original image overlaps the original mask.


In [ ]:
chest_left, tissue_share, sizes, whole = [], [], [], 0
for image in manifest.images:
    box = record.images[image.file_id].box
    whole += box.height * box.width > 0.95 * image.rows * image.cols
    pixels = read_image(image_path(OUT, image.file_id))
    tissue = pixels > settings.threshold
    third = pixels.shape[1] // 3
    chest_left.append(tissue[:, :third].mean() > tissue[:, -third:].mean())
    tissue_share.append(tissue.mean())
    sizes.append(pixels.shape)

flips = sum(params.flipped for params in record.images.values())
print(f"flipped: {flips} of {len(record.images)}")
print(f"chest wall on the left: {sum(chest_left)} of {len(chest_left)} images")
print(f"boxes over 95% of the image: {whole}")
print(
    f"tissue share of the preprocessed image: median {np.median(tissue_share):.2f}, "
    f"min {min(tissue_share):.2f}, max {max(tissue_share):.2f}"
)
print(
    "longer side:",
    sorted({max(s) for s in sizes}),
    "| height:",
    min(s[0] for s in sizes),
    "to",
    max(s[0] for s in sizes),
    "| width:",
    min(s[1] for s in sizes),
    "to",
    max(s[1] for s in sizes),
)

In [ ]:
def contrast(pixels: np.ndarray, mask: np.ndarray) -> float:
    inside = mask > 0
    ring = ndimage.binary_dilation(inside, iterations=15)
    ring &= ~ndimage.binary_dilation(inside, iterations=3)
    ring &= pixels > 0
    return float(pixels[inside].mean() - pixels[ring].mean())


brighter, overlaps = [], []
for image in manifest.images:
    pixels = read_image(image_path(OUT, image.file_id)).astype(np.float32)
    mask = read_mask(processed_mask_path(OUT, image.file_id))
    brighter.append(contrast(pixels, mask) > 0)
    restored = restore_mask(mask, record.images[image.file_id])
    original = read_mask(mask_path(MASKS, image.file_id))
    overlaps.append(
        ((restored == 1) & (original == 1)).sum()
        / ((restored == 1) | (original == 1)).sum()
    )

print(f"mask brighter than its surroundings: {sum(brighter)} of {len(brighter)} images")
print(
    f"restored mask vs original mask (IoU): min {min(overlaps):.3f}, "
    f"median {np.median(overlaps):.3f}"
)